# Proyecto final · Compras Claras — un caso, un producto para todos
**Universidad Central · Big Data 2026-2S · grupo 2 · proyecto de integración S01–S14**

**Producto:** aplicación `observatorio.html` para buscar y filtrar registros SECOP reales, consultar predicciones de modalidad administrativa y exportar resultados. Funciona **sin servidor y sin mantener un Codespace encendido**. No detecta fraude.

**Dos recursos del estudiante:** [Guía interactiva y consulta de S01–S14](https://jazaineam1.github.io/BigData2026/Presentaciones/proyecto-final-explorer.html) y **este cuaderno ejecutable**. Sigue E0→E7 en orden y completa las siete expresiones `____` con tu propio resultado.

**Primera opción:** reutiliza el Codespace **S13 que ya tengas** (2 núcleos; detenerlo cuando termines). **Si no tienes cuota:** usa Google Colab con CPU sin GPU; E0 obtiene cuatro archivos indispensables y **no clona todo el repositorio**. Trabaja con el mismo caso y evita instalaciones pesadas.

**Databricks Free:** descarga `secop_final.csv` después de E2, súbelo por la interfaz de Databricks y ejecuta E5–E6 allí. **No pagues una cuenta o máquina para realizar esta actividad.** La edición gratuita tiene cuotas; si se bloquea, conserva el error real y consulta la alternativa con el docente antes de presentar un resultado como ejecutado.

**Mismo problema, etiquetas, modelo y split para todos.** Dask distribuye los experimentos independientes; el ajuste Spark MLlib es otra técnica de entrenamiento. Se conserva un único conjunto de prueba.

**Fecha de entrega y peso institucional:** se comunicarán por el canal oficial del grupo 2; no supongas una fecha tomada del PDA de otro grupo.

## Rúbrica de 100 puntos
| Producto y evidencia verificable | Puntos |
|---|---:|
| Descarga HTTP paginada y comparación `OFFSET` | 8 |
| Calidad, ETL, fuente y partición train/test | 12 |
| Procesamiento Dask y workers observados | 12 |
| Entrenamiento de experimentos independientes con Dask | 13 |
| Databricks, SQL y tablas Delta | 15 |
| Entrenamiento Spark MLlib guiado | 15 |
| **Observatorio HTML funcional, filtros y exportación** | **20** |
| Manifiesto, trazas y sustentación | 5 |
| **Total** | **100** |

**Equidad:** no se exige pagar cuotas ni ejecutar benchmarks masivos. Si una plataforma gratuita no te permite continuar, conserva evidencia técnica auténtica y solicita la alternativa docente. Los datos sintéticos de pruebas no reemplazan las evidencias reales SECOP.

## Mapa de aprendizaje · un solo caso, sin etapas desechables
E0: preparar el entorno sin nuevos Codespaces → E1: OFFSET real y control de páginas → E2: dataset/SHA y división fija → E3: Dask DataFrame y workers → E4: tres entrenamientos independientes con Dask → E5: Databricks, Spark SQL y Delta → E6: un modelo Spark MLlib guiado → E7: observatorio HTML exportable y defensa.

**Cómo se aprende:** cada bloque tiene una demostración resuelta y una microintervención de **una expresión** con comprobación automática (E1.1, E2.1, E3.1, E4.1, E5.1, E6.1 y E7.1). Los espacios `____` son intencionales; el estudiante debe completarlos **con sus resultados**, no limitarse a pulsar “Ejecutar todo”. No se cambian fuente, modalidades, algoritmo ni división.

**Duración y cuotas:** reutiliza el Codespace S13 o Colab CPU; conserva CSV/JSON/HTML y detén el entorno al terminar. E5/E6 se ejecutan solo desde Databricks Free, sin acceso garantizado ni compra de cómputo. Si falla una plataforma, deja el error auténtico y sigue la contingencia docente.

**Publicación:** borrador docente. No está aprobado ni publicado en el portal ni LMS.

## E0 · Preparar el entorno sin gastar la cuota de GitHub
**HAZ ESTO:** antes de crear otro Codespace, revisa en GitHub **Settings → Billing → Usage** cuánto te queda. Si ya tienes abierto el perfil S13, reutilízalo. Selecciona dos núcleos y configura autoapagado corto. Con 120 horas-núcleo gratuitas, dos núcleos equivalen a un máximo teórico de 60 horas de reloj por mes, **menos lo ya utilizado**.

**SI YA NO TIENES CUOTA:** abre el cuaderno en **Google Colab**, opción **CPU** sin GPU. La celda E0 descargará solo \`analisis.py\`, los 1.000 SECOP y el corpus docente de 200 expedientes; no clona los ~375 MB del repositorio. Colab tiene límites variables, así que guarda evidencias por etapa.

**DEBE VERSE:** raíz mínima o completa, archivo SECOP de 1.000 filas y librerías listas. **EVIDENCIA:** SHA-256 de la fuente y entorno identificado.

**Cómo se lee:** se muestra un origen de archivo real.
**Qué nos dice:** el resto del recorrido no necesita otra descarga de datos.
**Qué NO permite concluir:** disponibilidad ilimitada de Codespaces, Colab ni Databricks.
**Qué error común:** crear tres máquinas y repetir instalaciones para un único taller.

In [ ]:
from pathlib import Path
import sys, os, hashlib, json, importlib.util
raiz = Path.cwd().resolve()
for candidato in [raiz, *raiz.parents]:
    if (candidato / "Laboratorios/S13_Codespaces/analisis.py").exists():
        raiz = candidato
        entorno = "Codespaces o entorno local con repo completo"
        break
else:
    # Plan B: Colab CPU. Descarga archivos indispensables, NO clona el repositorio.
    from urllib.request import urlopen
    raiz = Path.cwd().resolve() / "BigData2026_final_minimo"
    base_url = "https://raw.githubusercontent.com/jazaineam1/BigData2026/main/"
    archivos = [
        "Laboratorios/S13_Codespaces/analisis.py",
        "Cuadernos/datos/secop_chunks/prueba_chunk_0000000.csv",
        "Datos/tc2_corpus_2026.json",
        "utils/proyecto_final_offset.py",
    ]
    for relativo in archivos:
        destino = raiz / relativo
        destino.parent.mkdir(parents=True, exist_ok=True)
        if not destino.is_file():
            try:
                with urlopen(base_url + relativo, timeout=40) as fuente:
                    destino.write_bytes(fuente.read())
            except Exception as err:
                raise RuntimeError("Fallo al obtener "+relativo+"; verifica acceso a GitHub público") from err
    entorno = "Colab CPU / archivos mínimos"
if importlib.util.find_spec("dask") is None or importlib.util.find_spec("distributed") is None or importlib.util.find_spec("pyarrow") is None:
    import subprocess
    subprocess.check_call([sys.executable,"-m","pip","install","-q",
                           "dask[dataframe,distributed]>=2025.1,<2027","pyarrow>=17,<25"])
sys.path.insert(0, str(raiz / "Laboratorios/S13_Codespaces"))
sys.path.insert(0, str(raiz / "utils"))
import analisis as S13
directorio = raiz / "Laboratorios/Proyecto_Final_2026/salidas"
directorio.mkdir(parents=True, exist_ok=True)
assert S13.CSV.exists(), "Falta el CSV real de S13"
assert (raiz / "utils/proyecto_final_offset.py").exists(), "Falta helper OFFSET del taller"
print("ENTORNO:", entorno)
print("FUENTE:", S13.CSV.relative_to(raiz))
print("CPU lógicas visibles:", os.cpu_count())
print("CSV SHA256:", hashlib.sha256(S13.CSV.read_bytes()).hexdigest())
print("OK: material de S13 localizado.")

## E1 · Descarga SECOP II con OFFSET: reaplicar lo aprendido en TC1
**HAZ ESTO:** realiza dos páginas de veinte registros mediante `$offset=0` y `$offset=20`, primero secuencial y después concurrente. Usarás la **misma ventana histórica del TC1: enero-febrero de 2024**, exactamente el orden del contrato TC1, con desempate `:id ASC`. Esto reduce diferencias de fechas y permite comprobar que no pediste dos veces el mismo conjunto.

**DEBE VERSE:** en `A_descarga.json` los offsets [0,20], `limit` 20 por página, 40 filas en cada ejecución, huellas SHA-256 y tiempos observados. Si HTTP 429, se respetan reintentos acotados. Los RAW se guardan como `A_paginas_secuencial.json` y `A_paginas_paralelo.json`, jamás como cifras inventadas.

**SI FALLA:** el archivo registra el tipo de error; el pipeline continúa con el CSV congelado del S13, **no** con una simulación de descarga. Recuerda que Colab y Codespaces necesitan acceso de red. Si ambas descargas tienen 40 filas pero hash diferente, se indica que la fuente pudo cambiar entre lecturas y no se afirma equivalencia.

**EVIDENCIA:** A_descarga.json + páginas RAW auténticas cuando el servicio respondió. La fuente de entrenamiento permanece separada: `prueba_chunk_0000000.csv`.

**Para aprender el código:** abre `utils/proyecto_final_offset.py`, ubica `parametros`, `obtener_pagina`, `descargar` y `ejecutar`. Verifica `$select`, `$where`, `$order`, `$limit` y `$offset` igual que en TC1. El archivo tiene pruebas pequeñas que no consumen Internet: `python -m unittest -v tests/test_proyecto_final_offset.py`.

**Cómo se lee:** OFFSET es la posición inicial (basada en cero); LIMIT es el tamaño solicitado, y `:id` desempata el orden.
**Qué nos dice:** que realmente se pidieron páginas distintas, se ensamblaron en orden y se comparó su contenido.
**Qué NO permite concluir:** que la fuente es inmutable, que paralelizar siempre es más rápido o que estas páginas son la fuente congelada del modelo.
**Qué error común:** pedir offset=1 para la segunda página de 20 o comparar diferentes filtros/órdenes.

In [ ]:
# E1 · Reutilizamos el método OFFSET del TC1, con límites apropiados al proyecto.
# Código del método: utils/proyecto_final_offset.py
# La prueba real de la API se hace una sola vez por equipo; no al hacer Run All repetidamente.
import requests
import proyecto_final_offset as descarga_offset
from pathlib import Path
import inspect

print("Fuente oficial:", descarga_offset.ENDPOINT)
print("Filtro histórico TC1:", descarga_offset.WHERE)
print("Orden estable TC1:", descarga_offset.ORDER)
print("Páginas y LIMIT:", [descarga_offset.parametros(o) for o in (0,20)])
print("Función que prepara OFFSET:", inspect.getsource(descarga_offset.parametros))

# Para no repetir llamadas si ya existe un resultado en la sesión,
# reutiliza A_descarga.json (aun si hubo una falla registrada).
ruta_a = directorio / "A_descarga.json"
if ruta_a.exists():
    evidencia_a = json.loads(ruta_a.read_text(encoding="utf-8"))
    print("E1: se reutiliza evidencia existente. Borra A_descarga.json solo para repetir a conciencia.")
else:
    evidencia_a = descarga_offset.ejecutar(destino=directorio, get=requests.get)

print(json.dumps(evidencia_a, indent=2, ensure_ascii=False))

### E1.1 · Comprobación del OFFSET (1 expresión)
**Qué debes hacer:** determina el desplazamiento inicial de la segunda página, si cada página contiene 20 registros. Completa únicamente `____` en la próxima celda. **No vuelve a llamar a la API**.

**Pista:** la primera página comienza en cero y las páginas no deben solaparse. Busca el tamaño en `descarga_offset.PAGE_SIZE`.

**Comprobación:** verás las posiciones 0 y 20. **Qué nos dice:** se consultarán dos rangos diferentes. **Qué NO concluye:** que SECOP respondió, porque las pruebas de parámetros no son HTTP reales. **Error frecuente:** confundir el número de página con el desplazamiento de registros.

In [ ]:
# E1.1 · Una sola expresión. Sin Internet.
offset_segunda_pagina = ____
assert isinstance(offset_segunda_pagina, int) and not isinstance(offset_segunda_pagina, bool)
assert offset_segunda_pagina == descarga_offset.PAGE_SIZE
primera = descarga_offset.parametros(0)
segunda = descarga_offset.parametros(offset_segunda_pagina)
assert primera["$offset"] == 0 and segunda["$offset"] == 20
print("OFFSET solicitado:", primera["$offset"], segunda["$offset"])

## E2 · Preparar exactamente los mismos datos para Dask y Databricks
**HAZ ESTO:** filtra las dos modalidades conocidas y usa el conjunto preparado por el docente. En la fuente hay 19 descripciones `No Definido` y dos procesos con el mismo texto `auxiliar administrativo`; el cuaderno retira el marcador sin información y conserva una sola copia del segundo texto **antes de separar entrenamiento/prueba**, para no hacer pasar dos veces el mismo texto por conjuntos distintos. Esta limpieza está resuelta; no se califica como auditoría del estudiante. La división estratificada mantiene semilla 42 y una sola columna split compartida con Databricks.

**DEBE VERSE:** dos modalidades, IDs únicos y archivos secop_final.csv / B_fuente.json.

**SI FALLA:** verifica los nombres reales de columnas de S13; NO cambies etiquetas ni sustituyas por datos sintéticos.

**EVIDENCIA:** CSV y manifiesto con SHA-256.

**Cómo se lee:** train son las observaciones para ajustar el modelo; test son registros reservados.
**Qué nos dice:** ambos motores podrán ver exactamente la misma prueba.
**Qué NO permite concluir:** que un modelo tendrá buen rendimiento o que las etiquetas miden riesgo.
**Qué error común:** ajustar el TF-IDF usando test antes de separar los datos.

In [ ]:
import pandas as pd
tabla = S13.leer_etiquetados()       # verifica 1000 filas reales
# Curaduría docente: excluir descripción marcador y asegurar textos distintos antes del split.
# No inventar etiquetas ni duplicar registros para simular escala.
muestra_previa = S13.preparar_clasificacion(tabla)
texto_normalizado = (muestra_previa[S13.COL_TEXTO].astype(str)
                     .str.strip().str.lower().str.replace(r"\s+", " ", regex=True))
sin_marcador = muestra_previa.loc[texto_normalizado.ne("no definido")].copy()
sin_marcador["_clave_texto"] = (sin_marcador[S13.COL_TEXTO].astype(str)
                              .str.strip().str.lower().str.replace(r"\s+", " ", regex=True))
tabla_curada = sin_marcador.drop_duplicates("_clave_texto").drop(columns=["_clave_texto"]).copy()
muestra = S13.preparar_clasificacion(tabla_curada)
x_train,x_test,y_train,y_test = S13.separar_muestra(tabla_curada)
assert len(muestra) == 810, "La curaduría docente espera 810 textos únicos del snapshot S13"
splits = pd.Series("test",index=muestra.index)
splits.loc[x_train.index]="train"
base = pd.DataFrame({
    "id_del_proceso":muestra["id_del_proceso"].astype(str),
    "texto":muestra[S13.COL_TEXTO].astype(str),
    "modalidad":muestra[S13.COL_LABEL].astype(str),
    "split":splits
})
assert base["id_del_proceso"].is_unique
assert set(base["modalidad"])==set(S13.CLASES)
assert set(base["split"])=={"train","test"}
assert base.loc[base.split=="train","id_del_proceso"].isin(
       base.loc[base.split=="test","id_del_proceso"]).sum()==0
ruta_csv=directorio/"secop_final.csv"
base.to_csv(ruta_csv,index=False,encoding="utf-8")
evidencia_b={"fuente":str(S13.CSV.relative_to(raiz)),
 "sha256_fuente":hashlib.sha256(S13.CSV.read_bytes()).hexdigest(),
 "csv_salida":"secop_final.csv","sha256_salida":hashlib.sha256(ruta_csv.read_bytes()).hexdigest(),
 "filas_originales":int(len(tabla)),"filas_modelo":int(len(base)),
 "filas_elegibles_antes_curaduria":int(len(muestra_previa)),
 "marcadores_excluidos":int(len(muestra_previa)-len(sin_marcador)),
 "textos_duplicados_retirados":int(len(sin_marcador)-len(muestra)),
 "train":int((base.split=="train").sum()),
 "test":int((base.split=="test").sum()),
 "clases":base.modalidad.value_counts().to_dict(),"random_state":42}
(directorio/"B_fuente.json").write_text(json.dumps(
    evidencia_b,ensure_ascii=False,indent=2),encoding="utf-8")
print(json.dumps(evidencia_b,indent=2,ensure_ascii=False))

### E2.1 · Reto de código pequeño: comprobar una clase del test
Ya viste una solución completa de ETL en E2. **Ahora te toca escribir una sola expresión Python**, manteniendo el mismo caso y sin cambiar los datos.

**Instrucción:** sustituye únicamente `____` para contar cuántos procesos de **Contratación directa** hay en `test` utilizando `base`. Pista: `base[condicion1 & condicion2]`, `len(...)`. Cuando lo completes debe cumplirse el `assert` y debes anotar el número de tu ejecución.

**Resultado esperado:** un entero no negativo menor que el tamaño del test. Si falla, comprueba que cada comparación tiene paréntesis y usa `&` entre condiciones. No pegues respuestas de otro equipo.

**Cómo se lee:** son registros reales de esa clase reservados para probar el clasificador.
**Qué nos dice:** el universo sobre el que se evaluará esa modalidad.
**Qué NO permite concluir:** F1 ni representatividad nacional sin una comparación adicional.
**Qué error común:** contar todo el conjunto y presentarlo como test.

In [ ]:
# E2.1 · Completa SOLO una expresión. Mantén nombres y fuentes originales.
n_directa_en_test = ____   # Pista: len(base[(base["split"]=="test") & (... )])
assert isinstance(n_directa_en_test, (int, np.integer)) if "np" in globals() else isinstance(n_directa_en_test,int)
assert n_directa_en_test == int(((base["split"]=="test") &
                                (base["modalidad"]=="Contratación directa")).sum())
print("Contratación directa en test:",n_directa_en_test)

## E3 · Dask: procesamiento real de los mismos registros
**HAZ ESTO:** usa 2 workers de Dask Distributed con 1 hilo cada uno; calcula conteos por modalidad y comprueba la igualdad contra pandas. Los workers pueden ser hilos dentro de un mismo Codespace: no son computadores físicos diferentes.

**DEBE VERSE:** 2 workers observados y los mismos conteos.

**SI FALLA:** comprueba dask.distributed instalado y carga del entorno S13. No aumentes workers para intentar esconder errores.

**EVIDENCIA:** C_dask.json. El tiempo no es una calificación.

**Cómo se lee:** workers_reales es el número reconocido por el scheduler; particiones son bloques de datos.
**Qué nos dice:** Dask calculó el mismo resultado desde tareas particionadas.
**Qué NO permite concluir:** que el problema necesita un clúster o que Dask es siempre más rápido.
**Qué error común:** decir que Dask ejecutó al crear el grafo sin llamar compute().

In [ ]:
import dask.dataframe as dd
from distributed import Client
import time
inicio=time.perf_counter()
with Client(processes=False,n_workers=2,threads_per_worker=1,dashboard_address=None) as cliente:
    workers=cliente.scheduler_info()["workers"]
    ddf=dd.from_pandas(base,npartitions=4)
    conteos_dask={str(k):int(v) for k,v in
                  ddf.groupby("modalidad").size().compute().items()}
    trabajadores=[{"id":str(k),"hilos":int(v["nthreads"])}
                   for k,v in workers.items()]
tiempo=time.perf_counter()-inicio
conteos_pandas={str(k):int(v) for k,v in base.groupby("modalidad").size().items()}
assert conteos_dask==conteos_pandas and len(trabajadores)==2
evidencia_c={"workers_reales":trabajadores,"particiones":4,
 "conteos_dask":conteos_dask,"conteos_pandas":conteos_pandas,
 "coinciden":True,"segundos_con_inicio":round(tiempo,5),
 "nota":"Dask DataFrame distribuido; muestra pequeña sin garantía de aceleración."}
(directorio/"C_dask.json").write_text(json.dumps(
    evidencia_c,ensure_ascii=False,indent=2),encoding="utf-8")
print(json.dumps(evidencia_c,indent=2,ensure_ascii=False))

### E3.1 · ¿Dask produjo el mismo resultado? (1 expresión)
**Qué debes hacer:** escribe una comparación entre los dos diccionarios de conteos que produjo E3. No escribas un `True` fijo: utiliza las variables `conteos_dask` y `conteos_pandas`.

**Debes observar:** `True` y el número de workers. **Qué nos dice:** ambas herramientas responden la misma pregunta. **Qué NO permite concluir:** una mejora de rendimiento en una muestra pequeña. **Error frecuente:** comparar únicamente el tiempo e ignorar la corrección.

In [ ]:
# E3.1 · Reutiliza las salidas verdaderas de la etapa anterior.
coinciden_los_resultados = ____
assert coinciden_los_resultados is True
assert coinciden_los_resultados == (conteos_dask == conteos_pandas)
print("¿Mismos conteos?", coinciden_los_resultados,
      "· workers observados:", len(trabajadores))

## E4 · Entrenamiento de tres modelos independientes con Dask
**HAZ ESTO:** aprovecha el laboratorio S13. Entrena regresión logística con TF-IDF, valores de C iguales para todos: 0.1, 1.0 y 10.0. La validación cruzada usa solamente train. El test se utiliza una vez al final.

**DEBE VERSE:** tres valores de C y sus F1, dos workers observados y una F1 final. Contrasta contra el clasificador de referencia DummyClassifier.

**SI FALLA:** usa la muestra real original, revisa memoria y comprueba el módulo S13. No ejecutes benchmarks de 50.000.

**EVIDENCIA:** D_modelo_dask.json.

**Cómo se lee:** F1 armoniza precisión y cobertura; la métrica representa estas dos clases y este test.
**Qué nos dice:** si el clasificador supera o no al control sencillo.
**Qué NO permite concluir:** generalización a cualquier contrato futuro, ni entrenar un único modelo repartido internamente.
**Qué error común:** elegir el hiperparámetro usando el test.

In [ ]:
# E4 · 3 experimentos independientes coordinados por Dask
# Reutiliza funciones ya enseñadas en S13: el test NUNCA decide el valor de C.
referencia = S13.entrenar_y_evaluar(tabla_curada)
resultado_dask = S13.experimentos_distribuidos(tabla_curada, n_workers=2)
assert len(resultado_dask["configuraciones"]) == 3
assert resultado_dask["workers_en_un_codespace"] == 2
evidencia_d = {
    "f1_referencia": referencia["f1_macro"],
    "f1_dummy": referencia["f1_control"],
    "matriz_confusion": referencia["matriz_confusion"],
    "workers": resultado_dask["workers_en_un_codespace"],
    "configuraciones": resultado_dask["configuraciones"],
    "mejor_C": resultado_dask["mejor_C"],
    "f1_prueba_final": resultado_dask["f1_prueba_final"],
    "significado": "Dask reparte tres experimentos; cada fit scikit-learn es local.",
}
(directorio / "D_modelo_dask.json").write_text(
    json.dumps(evidencia_d, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(json.dumps(evidencia_d, indent=2, ensure_ascii=False))

### E4.1 · Elegir el modelo sin consultar el test (1 expresión)
**Qué debes hacer:** de las tres configuraciones registradas en `resultado_dask["configuraciones"]`, selecciona el valor de `C` que tenga mayor `f1_cv`. Completa `____` usando el resultado del entrenamiento.

**Pista:** busca la clave `"mejor_C"` en `resultado_dask`. La selección se efectuó mediante validación cruzada **sobre train**; el test se evalúa después. **Qué NO permite concluir:** que esa configuración sea óptima para todos los contratos futuros. **Error frecuente:** elegir C por la puntuación de test.

In [ ]:
# E4.1 · Selecciona un valor ya calculado en tu ejecución, no un número copiado.
C_seleccionado = ____
assert float(C_seleccionado) == float(max(
    resultado_dask["configuraciones"], key=lambda x: x["f1_cv"]
)["C"])
assert C_seleccionado == evidencia_d["mejor_C"]
print("Mejor C de validación cruzada:", C_seleccionado)

## E5 · Databricks Free: archivo local → tabla Delta → Spark SQL
**CAMBIA DE ENTORNO.** Descarga secop_final.csv del Codespace y súbelo manualmente en Databricks Free mediante **Catalog → Create/Modify table from upload**. El menú puede cambiar; busca la carga de CSV. No intentes descargar desde SECOP en Databricks, porque el acceso saliente está restringido.

1. Comprueba en Databricks **Environment** una versión compatible con Spark ML en serverless (versión 4 o posterior, si aparece). Si no está disponible, registra la limitación sin instalar ni pagar recursos. Crea una tabla Delta a partir del CSV. Conserva nombres id_del_proceso, texto, modalidad, split.
2. Abre este mismo cuaderno en Databricks y **ejecuta E5/E6 solamente**. Debes editar TABLA_SUBIDA con el nombre real en tu entorno (catálogo.esquema.tabla). No es una elección académica.
3. La celda crea tablas Bronze, Silver, Gold con Spark SQL/Delta sin añadir tecnologías no vistas.
4. La tabla Gold muestra cuántos procesos hay por clase y partición; deben concordar con E2.

**DEBE VERSE:** 3 tablas, consulta, conteos y JSON E_databricks.
**SI FALLA:** verifica nombre real de catálogo/esquema, carga CSV y cuotas del plan. Comunica el fallo al docente antes de concluir.
**EVIDENCIA:** salida JSON E_databricks, captura real y consulta SQL.

**Cómo se lee:** Bronze preserva origen; Silver valida columnas; Gold agrega.
**Qué nos dice:** los registros pueden procesarse y consultarse en plataforma Spark.
**Qué NO permite concluir:** que se construyó un clúster personalizado ni que se realizó entrenamiento.
**Qué error común:** usar rutas relativas del Codespace en Databricks o volver a dividir train/test.

**Configuración imprescindible de CSV:** en Advanced attributes activa `Rows span multiple lines` (viene desactivada) y desactiva la detección automática de tipos; las cuatro columnas son texto. En SECOP hay descripciones que abarcan varias líneas y no deben partirse. Si no logras una importación fiel, en el entorno E2 ejecuta `base.to_parquet(directorio / "secop_final.parquet", index=False)` y sube ese Parquet por la misma interfaz. Conserva `secop_final.csv` como evidencia original; el Parquet es solo una alternativa de transporte con las mismas filas y split. No es una nueva entrega. [Documentación de importación](https://docs.databricks.com/aws/en/ingestion/create-or-modify-table).

**Antes de ejecutar:** en Workspace importa el IPYNB y en Environment selecciona versión 4 o posterior compatible con MLlib. No ejecutes E0–E4 en el kernel remoto. Crea una tabla de importación propia con un nombre distinto por equipo y copia su nombre de tres niveles al bloque E5. Ejecuta E5, E5.1, E6 y E6.1 en ese orden. Los nombres de tablas derivados incluyen un sufijo de la tabla importada para evitar sobrescrituras entre equipos.

In [ ]:
# E5 · EJECUTAR SOLO EN UN NOTEBOOK DATABRICKS DESPUÉS DE SUBIR EL CSV.
# Sustituye únicamente el nombre de TABLA_SUBIDA por el que muestra Catalog.
TABLA_SUBIDA = "workspace.default.secop_final"
if "dbutils" not in globals():
    print("E5 NO EJECUTADO: este bloque corresponde a Databricks, no a Codespaces.")
else:
    from pyspark.sql import functions as F
    import json
    datos_spark=spark.table(TABLA_SUBIDA)
    obligatorias={"id_del_proceso","texto","modalidad","split"}
    assert obligatorias.issubset(set(datos_spark.columns)), "Revisa encabezados del CSV subido"
    assert datos_spark.count() == 810, "CSV: revisa Rows span multiple lines o importa el Parquet equivalente"
    assert datos_spark.select("id_del_proceso").distinct().count() == 810, "IDs de importación alterados"
    import hashlib
    prefijo=TABLA_SUBIDA.rsplit(".",1)[0]
    sufijo_tabla=hashlib.sha256(TABLA_SUBIDA.encode("utf-8")).hexdigest()[:10]
    BRONZE=prefijo+".uc_final_bronze_"+sufijo_tabla
    SILVER=prefijo+".uc_final_silver_"+sufijo_tabla
    GOLD=prefijo+".uc_final_gold_"+sufijo_tabla
    datos_spark.write.format("delta").mode("overwrite").saveAsTable(BRONZE)
    spark.sql(f"""
    CREATE OR REPLACE TABLE {SILVER} USING DELTA AS
    SELECT CAST(id_del_proceso AS STRING) AS id_del_proceso,
           CAST(texto AS STRING) AS texto,
           CAST(modalidad AS STRING) AS modalidad,
           CAST(split AS STRING) AS split
    FROM {BRONZE}
    WHERE texto IS NOT NULL AND TRIM(texto) <> ''
      AND split IN ('train','test')
      AND modalidad IN ('Contratación directa','Contratación régimen especial')
    """)
    spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD} USING DELTA AS
    SELECT modalidad,split,COUNT(*) AS registros
    FROM {SILVER} GROUP BY modalidad,split
    """)
    assert spark.table(SILVER).count()==datos_spark.count(), "E5: se perdieron filas"
    assert spark.table(SILVER).select("id_del_proceso").distinct().count()==datos_spark.count(), "E5: IDs duplicados"
    filas_gold=[r.asDict() for r in spark.table(GOLD).orderBy("modalidad","split").collect()]
    evidencia_e={"ejecucion":"DATABRICKS_REAL","tabla_subida":TABLA_SUBIDA,
                 "bronze":BRONZE,"silver":SILVER,"gold":GOLD,
                 "filas":datos_spark.count(),"conteos":filas_gold}
    print("E_DATABRICKS_JSON="+json.dumps(evidencia_e,ensure_ascii=False))
    display(spark.table(GOLD))

### E5.1 · Comprobar la tabla Gold (1 expresión, solo Databricks)
**Qué debes hacer:** suma el campo `registros` de las filas de `filas_gold`. Verifica que coincide con los registros importados. La consulta no descarga nada y no crea una tabla adicional.

**Qué nos dice:** el agregado Gold conserva todas las filas Silver. **Qué NO concluye:** que los textos sean correctos ni que la etiqueta prediga fraude. **Error frecuente:** sumar solo una modalidad o un solo conjunto. Si todavía estás en Codespaces o Colab, esta celda se salta sin afirmar éxito.

In [ ]:
# E5.1 · Una expresión solo cuando E5 se ejecutó de verdad en Databricks.
if "evidencia_e" not in globals():
    print("E5.1 pendiente: primero ejecutar E5 en Databricks Free.")
else:
    total_gold = ____
    assert isinstance(total_gold, int)
    assert total_gold == evidencia_e["filas"]
    print("Registros conciliados en Gold:", total_gold)
    import json
    evidencia_e["microreto_E5.1"] = total_gold
    print("E_DATABRICKS_JSON="+json.dumps(evidencia_e, ensure_ascii=False))

## E6 · Entrenamiento Spark MLlib guiado: mismo caso y mismo split
**Este bloque no se introdujo como práctica principal en S14:** el código está resuelto para que la exigencia evalúe la integración y lectura de evidencias, no memorizar una API nueva.

**Compatibilidad verificada documentalmente:** Databricks admite `pyspark.ml` en serverless con un entorno compatible (versión 4 o posterior). Esta declaración no sustituye la ejecución de prueba en una cuenta Free.\n\nPipeline: Tokenizer separa palabras → HashingTF crea características numéricas → IDF reduce peso de términos frecuentes → LogisticRegression ajusta una clasificación binaria. La representación difiere del TfidfVectorizer de scikit-learn, por lo que NO se exige igualdad exacta de F1 entre motores. Sí se exige **el mismo conjunto test**, que ya viene etiquetado en la columna split.

**HAZ ESTO:** después de E5, ejecuta en Databricks. El entrenamiento usa `repartition(2)` para visualizar dos particiones del conjunto train; `spark_partition_id()` permite contar sus registros sin acceder a la API RDD, restringida en serverless. **Dos particiones no son necesariamente dos máquinas ni una garantía de mayor velocidad.** No renombres las dos etiquetas ni vuelvas a dividir los datos.

**DEBE VERSE:** predicciones, F1 macro y matriz de confusión en test. **SI FALLA:** revisar entorno, tabla Silver, permisos o cuota del servicio. **EVIDENCIA:** F_spark_modelo como salida estructurada.

**Cómo se lee:** la matriz distingue modalidad real frente a predicha.
**Qué nos dice:** el comportamiento de este clasificador con estos registros.
**Qué NO permite concluir:** que predice fraude ni que 1.000 filas demuestran escalabilidad masiva.
**Qué error común:** entrenar sobre test o afirmar que los dos algoritmos usan características idénticas.

In [ ]:
# E6 · EJECUTAR SOLO EN DATABRICKS DESPUÉS DE E5.
if "dbutils" not in globals():
    print("E6 NO EJECUTADO: requiere el cuaderno remoto Databricks y E5 correcto.")
else:
    from pyspark.sql import functions as F
    from pyspark.ml import Pipeline
    from pyspark.ml.feature import Tokenizer, HashingTF, IDF
    from pyspark.ml.classification import LogisticRegression
    from pyspark.ml.evaluation import MulticlassClassificationEvaluator
    datos_ml=(spark.table(SILVER)
      .withColumn("label",F.when(F.col("modalidad")=="Contratación directa",F.lit(0.0)).otherwise(F.lit(1.0))))
    # Dos particiones explícitas: observables en un solo cómputo de clase.
    # No equivalen a dos máquinas ni prueban mejora de rendimiento.
    train_ml=datos_ml.filter(F.col("split")=="train").repartition(2)
    test_ml=datos_ml.filter(F.col("split")=="test")
    assert train_ml.count()>0 and test_ml.count()>0
    reparto=[r.asDict() for r in
        train_ml.select(F.spark_partition_id().alias("particion"))
                .groupBy("particion").count().orderBy("particion").collect()]
    assert sum(int(r["count"]) for r in reparto)==train_ml.count()
    print("Particiones Spark observadas en entrenamiento:", reparto)
    tokenizador=Tokenizer(inputCol="texto",outputCol="tokens")
    tf=HashingTF(inputCol="tokens",outputCol="tf",numFeatures=4096)
    idf=IDF(inputCol="tf",outputCol="features",minDocFreq=2)
    logistica=LogisticRegression(featuresCol="features",labelCol="label",maxIter=25,regParam=.1)
    pipeline=Pipeline(stages=[tokenizador,tf,idf,logistica])
    modelo_spark=pipeline.fit(train_ml)   # entrenamiento con cálculos de Spark sobre particiones
    pred=modelo_spark.transform(test_ml)
    conteos={(float(r["label"]),float(r["prediction"])):int(r["count"])
       for r in pred.groupBy("label","prediction").count().collect()}
    f1_por_clase=[]
    for cls in (0.0,1.0):
        vp=conteos.get((cls,cls),0)
        fp=sum(n for (real,estim),n in conteos.items() if estim==cls and real!=cls)
        fn=sum(n for (real,estim),n in conteos.items() if real==cls and estim!=cls)
        f1_por_clase.append((2*vp/(2*vp+fp+fn)) if 2*vp+fp+fn else 0.)
    F1_MACRO=round(sum(f1_por_clase)/2,4)
    pred.select("id_del_proceso","label","prediction").write.format("delta").mode("overwrite").saveAsTable(
        prefijo+".uc_final_predicciones_"+sufijo_tabla)
    evidencia_f={"ejecucion":"SPARK_MLLIB_REAL","entrenamiento":train_ml.count(),
      "prueba":test_ml.count(),"f1_macro":F1_MACRO,
      "particiones_entrenamiento":reparto,
      "f1_por_clase":[round(x,4) for x in f1_por_clase],
      "matriz_real_0_1_pred_0_1":[
         [conteos.get((0.0,0.0),0),conteos.get((0.0,1.0),0)],
         [conteos.get((1.0,0.0),0),conteos.get((1.0,1.0),0)]],
      "tabla_predicciones":prefijo+".uc_final_predicciones_"+sufijo_tabla,
      "limitacion":"No equivale numéricamente a sklearn; representa clases administrativas."}
    print("F_SPARK_MODELO_JSON="+json.dumps(evidencia_f,ensure_ascii=False))
    display(pred.select("id_del_proceso","modalidad","prediction").limit(10))

### E6.1 · Leer una matriz de confusión (1 expresión, solo Databricks)
**Qué debes hacer:** suma las dos celdas fuera de la diagonal de `evidencia_f["matriz_real_0_1_pred_0_1"]`. Esos son los errores de clasificación observados en test.

**Qué nos dice:** la cantidad de procesos mal clasificados en esta prueba. **Qué NO concluye:** que el proceso contractual presente irregularidades. **Error frecuente:** contar aciertos de la diagonal como errores. En Codespaces o Colab se conserva como pendiente, sin inventar predicciones.

In [ ]:
# E6.1 · Comprueba la matriz producida por Spark en el test real.
if "evidencia_f" not in globals():
    print("E6.1 pendiente: requiere E6 en Databricks Free.")
else:
    matriz_test = evidencia_f["matriz_real_0_1_pred_0_1"]
    errores_spark = ____
    assert errores_spark == matriz_test[0][1] + matriz_test[1][0]
    assert 0 <= errores_spark <= evidencia_f["prueba"]
    print("Errores del test en Spark MLlib:", errores_spark)
    import json
    evidencia_f["microreto_E6.1"] = errores_spark
    print("F_SPARK_MODELO_JSON="+json.dumps(evidencia_f, ensure_ascii=False))

## E7 · Construir el producto: Observatorio SECOP que funciona sin Codespaces
**Este paso convierte el pipeline en una herramienta.** El estudiante no entrega solo JSON: genera su propia aplicación estática con registros, filtros, búsqueda y predicciones del conjunto test.

**HAZ ESTO:** en el entorno E0, ajusta una regresión logística con el mejor C elegido en E4, usando **solo train**; genera \`predicciones.csv\` para test, con ID real, etiqueta real, etiqueta predicha y probabilidad. Después usa el generador didáctico provisto para producir \`observatorio.html\`. El código del generador está en \`Laboratorios/Proyecto_Final_2026/generar_observatorio.py\`: puedes abrirlo y comprender cada función.

**DEBE VERSE:** una página que abre localmente sin conexión, filtra modalidades, busca en descripciones reales y permite descargar los resultados filtrados. Comprueba que **las predicciones aparecen solo en test** y que el indicador F1 coincide con D_modelo_dask.json. Prueba al menos una consulta que encuentre resultados y otra que no encuentre.

**RETO EVALUABLE — HAZ, NO SOLO LEAS:** en tu aplicación, busca un contrato que el modelo predijo correctamente y otro incorrectamente (si existe); conserva sus **IDs reales** en el manifiesto. Identifica con una opción estructurada la limitación correcta: «no representa fraude» / «predice fraude» / «todos los resultados son exactos». No se pide redactar texto libre.

**SI FALLA:** si no se encuentra el generador al ejecutar desde Colab, se descarga solo ese archivo público; si falta \`predicciones.csv\`, ejecuta las celdas E2 y E4 antes de E7. El HTML funciona sin Streamlit, sin GitHub Pages y sin publicar enlaces.

**EVIDENCIA:** \`predicciones.csv\`, \`observatorio.html\`, \`manifest_final.json\`, consulta de prueba y PR en el repositorio privado existente. Si E5/E6 fueron ejecutadas, incorpora sus JSON reales desde Databricks, no pongas valores simulados.

**Cómo se lee:** al consultar una fila se ve modalidad observada y predicción sobre test.
**Qué nos dice:** el producto integra datos, modelo y recuperación consultable.
**Qué NO permite concluir:** calidad de predicciones sobre casos nuevos ni funcionamiento de un servicio SECOP en tiempo real.
**Qué error común:** mostrar predicciones de entrenamiento como si fueran evaluación externa.

**Traslado de evidencias sin credenciales:** después de resolver E5.1 y E6.1, copia únicamente el objeto JSON que aparece después de `E_DATABRICKS_JSON=` y `F_SPARK_MODELO_JSON=` en la última salida de cada microreto. Guarda cada objeto como `E_databricks.json` y `F_spark_modelo.json` (UTF-8, sin el prefijo) en `Laboratorios/Proyecto_Final_2026/salidas/` de tu entorno E0. En Colab usa Archivos → Subir; en Codespaces abre la carpeta y crea los dos archivos. No edites cifras ni inventes archivos si Databricks falló. Si ya ejecutaste E7, vuelve a ejecutar E7 y el cierre después de copiar. El cierre incorpora los microretos remotos al manifiesto.

In [ ]:
# E7 · Producto propio, autocontenido y consultable offline.
import numpy as np
mejor_C=float(evidencia_d["mejor_C"])
modelo_final=S13.nuevo_modelo(mejor_C)
modelo_final.fit(x_train,y_train)           # Exclusivamente entrenamiento
nombres_pred=modelo_final.predict(x_test)  # Exclusivamente prueba
confianzas=modelo_final.predict_proba(x_test).max(axis=1)
predicciones=pd.DataFrame({
    "id_del_proceso":base.loc[x_test.index,"id_del_proceso"].to_numpy(),
    "modalidad_real":y_test.to_numpy(),
    "modalidad_predicha":nombres_pred,
    "probabilidad_clase_predicha":np.round(confianzas,4),
})
assert len(predicciones)==len(x_test)
assert predicciones["id_del_proceso"].is_unique
ruta_pred=directorio/"predicciones.csv"
predicciones.to_csv(ruta_pred,index=False,encoding="utf-8")
from sklearn.metrics import f1_score
f1_producto=round(float(f1_score(y_test,nombres_pred,average="macro")),4)
assert abs(f1_producto-float(evidencia_d["f1_prueba_final"]))<0.00011, (
    "Las predicciones deben coincidir con el mejor C del bloque E4.")
generador = raiz / "Laboratorios/Proyecto_Final_2026/generar_observatorio.py"
if not generador.exists():
    # Ruta Colab: solo este archivo auxiliar de 11 KB; no clonar GitHub.
    from urllib.request import urlopen
    generador.parent.mkdir(parents=True,exist_ok=True)
    url = ("https://raw.githubusercontent.com/jazaineam1/BigData2026/main/"
           "Laboratorios/Proyecto_Final_2026/generar_observatorio.py")
    with urlopen(url,timeout=40) as fuente:
        generador.write_bytes(fuente.read())
sys.path.insert(0,str(generador.parent))
from generar_observatorio import generar
producto=generar(directorio/"secop_final.csv",ruta_pred,
                directorio/"D_modelo_dask.json",directorio/"observatorio.html")
esperados=["A_descarga.json","B_fuente.json","C_dask.json","D_modelo_dask.json",
           "secop_final.csv","predicciones.csv","observatorio.html",
           "E_databricks.json","F_spark_modelo.json"]
estado={}
for nombre in esperados:
    ruta=directorio/nombre
    estado[nombre]={"estado":"presente" if ruta.is_file() else "pendiente"}
    if ruta.is_file():
        estado[nombre]["sha256"]=hashlib.sha256(ruta.read_bytes()).hexdigest()
        estado[nombre]["bytes"]=ruta.stat().st_size
aciertos=(predicciones["modalidad_real"]==predicciones["modalidad_predicha"])
ids_correctos=predicciones.loc[aciertos,"id_del_proceso"].head(1).tolist()
ids_errores=predicciones.loc[~aciertos,"id_del_proceso"].head(1).tolist()
manifiesto={"curso":"BigData 2026-2S","taller":"Proyecto final Compras Claras",
 "mismo_caso_todos":True,"fuente_entrenamiento":"snapshot SECOP S13",
 "modelo":"TF-IDF + LogisticRegression, mejor C en validación cruzada",
 "f1_test":f1_producto,"ejemplo_acierto_id":ids_correctos,
 "ejemplo_error_id":ids_errores,"limitacion_escogida":"no representa fraude",
 "producto":producto,"estados":estado,
 "nota":"Presencia de archivos no significa validación remota en Databricks."}
(directorio/"manifest_final.json").write_text(json.dumps(
    manifiesto,ensure_ascii=False,indent=2),encoding="utf-8")
print("PRODUCTO LOCAL:",directorio/"observatorio.html")
print("F1 TEST:",f1_producto,"· Test:",len(predicciones))
print(json.dumps(manifiesto,ensure_ascii=False,indent=2))

### E7.1 · Auditar una predicción del producto (1 expresión)
**Qué debes hacer:** cuenta las filas de `predicciones` donde `modalidad_real` es distinta de `modalidad_predicha`. Después abre `observatorio.html` y busca un ID de ese grupo, si existe.

**Qué nos dice:** cuántas predicciones fallaron en el test reservado. **Qué NO concluye:** que el modelo detecta fraude o que la muestra representa todas las contrataciones. **Error frecuente:** confundir acierto con clasificación administrativa incorrecta.

In [ ]:
# E7.1 · El alumno debe utilizar las predicciones de su propio CSV test.
errores_del_producto = ____
assert isinstance(errores_del_producto, (int, np.integer))
assert errores_del_producto == int(
    (predicciones["modalidad_real"] != predicciones["modalidad_predicha"]).sum()
)
print("Fallos en test:", errores_del_producto,
      "de", len(predicciones), "procesos.")

In [ ]:
# Cierre · una sola traza de los siete microretos, sin repetir modelos ni descargar nada.
# Las etapas ejecutadas en Databricks se incluyen solo cuando hay evidencia auténtica.
# Los dos kernels no comparten memoria; recuperar solo evidencias transferidas.
for nombre, clave, variable in (
    ("E_databricks.json", "microreto_E5.1", "total_gold"),
    ("F_spark_modelo.json", "microreto_E6.1", "errores_spark"),
):
    ruta_remota = directorio / nombre
    if ruta_remota.is_file():
        remota = json.loads(ruta_remota.read_text(encoding="utf-8"))
        if clave in remota:
            assert type(remota[clave]) is int and remota[clave] >= 0, "Microreto remoto inválido"
            if nombre == "E_databricks.json":
                assert remota[clave] == remota["filas"], "Gold no concilia"
            else:
                assert remota[clave] <= remota["prueba"], "Errores fuera de rango"
            globals()[variable] = remota[clave]
resultados_microretos = {
    "E1.1_offset": int(offset_segunda_pagina) if "offset_segunda_pagina" in globals() else None,
    "E2.1_directa_en_test": int(n_directa_en_test) if "n_directa_en_test" in globals() else None,
    "E3.1_conteos_iguales": bool(coinciden_los_resultados) if "coinciden_los_resultados" in globals() else None,
    "E4.1_C_elegido_por_CV": float(C_seleccionado) if "C_seleccionado" in globals() else None,
    "E5.1_registros_Gold": int(total_gold) if "total_gold" in globals() else None,
    "E6.1_errores_Spark": int(errores_spark) if "errores_spark" in globals() else None,
    "E7.1_errores_observatorio": int(errores_del_producto) if "errores_del_producto" in globals() else None,
}
obligatorios = ["E1.1_offset", "E2.1_directa_en_test",
                "E3.1_conteos_iguales", "E4.1_C_elegido_por_CV",
                "E7.1_errores_observatorio"]
faltantes = [k for k in obligatorios if resultados_microretos[k] is None]
if faltantes:
    raise RuntimeError("Faltan microretos del entorno Codespaces/Colab: " + ", ".join(faltantes))
(directorio / "H_microretos.json").write_text(
    json.dumps(resultados_microretos, ensure_ascii=False, indent=2), encoding="utf-8")
manifiesto["microretos"] = resultados_microretos
manifiesto["estados"]["H_microretos.json"] = {
    "estado": "presente",
    "sha256": hashlib.sha256((directorio / "H_microretos.json").read_bytes()).hexdigest(),
}
(directorio / "manifest_final.json").write_text(
    json.dumps(manifiesto, ensure_ascii=False, indent=2), encoding="utf-8")
print("Producto listo para revisión: observatorio.html + manifest_final.json")
print("Microretos completados en este entorno:", len(obligatorios), "/ 5")
print("Databricks E5/E6:",
      "ejecutado aquí" if resultados_microretos["E5.1_registros_Gold"] is not None
      else "pendiente de evidencia externa real")

## Cierre · lo que realmente construiste
| Herramienta | Para qué se usó | Qué se demuestra |
|---|---|---|
| ThreadPoolExecutor | comparar dos páginas de una API | concurrencia HTTP ≠ datos mayores ni rapidez garantizada |
| CSV / ETL | guardar train/test con semilla y fuente | reproducibilidad |
| Dask DataFrame | procesamiento de particiones | scheduler, worker y \`compute()\` |
| Dask Distributed + scikit-learn | tres ajustes independientes | repartir experimentos ≠ un único fit distribuido |
| Spark SQL / Delta | consultas y capas analíticas | datos compartidos en Databricks |
| Spark MLlib | ajustar un modelo en Spark | un entrenamiento con cómputos distribuidos |
| Observatorio HTML propio | búsqueda, filtros, predicciones y descarga | producto útil **sin servidor ni Codespaces encendido** |

**No se califica:** tener más núcleos, menor tiempo, una cuenta paga, más registros duplicados, gastar Actions ni ejecutar 50.000 experimentos.

**Se califica:** datos verificables, decisiones con su propia salida, modelo interpretado con límites, consulta que funciona en el navegador y defensa del producto. La aplicación no hace inferencia en vivo sobre textos nuevos: muestra predicciones genuinas del test y consulta todos los registros existentes.

**Publicación:** guía disponible en la portada y en el módulo final del curso. La fecha y el peso oficiales se anunciarán por correo institucional después de confirmarlos para el grupo 2.